In [1]:
import pandas as pd

In [2]:
df_ground_truth = pd.read_csv('ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

# Running RAG

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [8]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join the course even if you just found out about it. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [9]:
assistant.total_cost()

0.00050475

In [10]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [11]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Can I still join the course if I just found out about it?',
 'answer_llm': 'Yes — you can still join the course even if you just found out about it. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
assistant.total_cost()

0.0009735

In [15]:
assistant.reset_usage()

In [16]:
assistant.total_cost()

0.0

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [19]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth[:-100], generate_rag_answer)

  0%|          | 0/620 [00:00<?, ?it/s]

In [20]:
df_results = pd.DataFrame(results)
df_results.head()

,question,answer_llm,answer_orig,document
0,Can I still join the course if I just found ou...,"Yes, you can still join the course. If you wan...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Is it too late to start this course now?,You can start whenever you want. The videos an...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join late, can I still get a certificate ...","Yes, but only if you complete the capstone pro...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to be eligible for the ce...,"To be eligible for the certificate, you need t...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,Are project submissions still open for new stu...,"Yes — if the submission form is still open, ne...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [24]:
assistant.total_cost()

1.4406555000000005

In [23]:
df_results.to_csv("rag_results.csv", index=False)